# 03 — Random Forest Classification
**Project:** SARS-CoV-2 Genomic Signatures — Explainable Analysis  
**Author:** Ibrahim Mustafa (Bembo)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print('Libraries loaded!')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/your_file.csv', sep='\t')
df['Clade'] = df['Clade'].str.replace('.', '', regex=False).str.strip()

X = df.drop(columns=['ID', 'Clade'])
y = df['Clade']

In [ ]:
# Sample 50k for training
sample_idx = df.sample(n=50000, random_state=42).index
X_rf = X.loc[sample_idx]
y_rf = y.loc[sample_idx]

X_train, X_test, y_train, y_test = train_test_split(
    X_rf, y_rf, test_size=0.2, random_state=42, stratify=y_rf)

print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

In [ ]:
# Train Random Forest
print('Training Random Forest...')
start = time.time()
rf = RandomForestClassifier(n_estimators=100, random_state=42,
                             n_jobs=-1, class_weight='balanced')
rf.fit(X_train, y_train)
print(f'Training time: {time.time()-start:.1f}s')

y_pred = rf.predict(X_test)
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=rf.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=[c.replace('Clade_','') for c in rf.classes_])

fig, ax = plt.subplots(figsize=(10, 8))
disp.plot(ax=ax, cmap='Blues', colorbar=True, xticks_rotation=45)
plt.title('Confusion Matrix — Random Forest')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# Feature Importance
feat_imp = pd.Series(rf.feature_importances_, index=X.columns)
feat_imp_sorted = feat_imp.sort_values(ascending=False)

plt.figure(figsize=(12, 6))
feat_imp_sorted.head(20).plot(kind='bar', color='steelblue')
plt.title('Top 20 Most Important Genomic Features (Random Forest)')
plt.xlabel('k-mer')
plt.ylabel('Importance')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

print('Top 10 features:')
print(feat_imp_sorted.head(10))

# Save model
import pickle
with open('rf_model.pkl', 'wb') as f:
    pickle.dump(rf, f)
print('\nModel saved as rf_model.pkl')